# ComfyUI — NTR MIX XIII + Bleach Watercolor / Rangiku

Notebook này chạy [ComfyUI](https://github.com/comfyanonymous/ComfyUI) trên GPU T4 miễn phí của Google Colab.

**Mặc định mới** (đổi model vì WAI v15 dễ ra nét bóng kiểu tô màu máy):

- Model nền **[NTR MIX XIII](https://civitai.com/models/926443/ntr-mix-or-illustrious-xl-or-noob-xl?modelVersionId=1166878)** (Illustrious, `NTRMIX XIII2.safetensors`, ~6,46 GB). Đây là model trong công thức nét Kubo bạn gửi, tác giả ghi là áp LoRA tốt. Vẫn có tùy chọn **[WAI-illustrious v15](https://civitai.com/models/827184/wai-illustrious-sdxl?modelVersionId=2167369)** nếu bạn muốn giữ bộ cũ.
- LoRA nét **[Bleach Watercolor](https://civitai.com/models/2440678/bleach-watercolor-style-tite-kubo-illustrious?modelVersionId=2744203)** — từ kích hoạt `bleach_watercolor`, weight 0.85
- LoRA nét **[Tite Kubo Style + drip art](https://civitai.com/models/1997021/tite-kubo-style-lora-drip-art?modelVersionId=2260426)** — từ kích hoạt `Tite_kubo_style`, weight 0.55 (nặng 1,27 GB, tắt được nếu hết RAM)
- LoRA nhân vật **[Rangiku Matsumoto Illustrious](https://civitai.com/models/1411559/rangiku-matsumoto-bleach-sdxl-lora-illustrious?modelVersionId=1595729)** — từ kích hoạt `rgku`, weight 0.85

Mỗi LoRA đều có ô bật/tắt. File không bật thì **không tải và không nạp**, để đỡ nặng GPU T4.

**Thứ tự trên điện thoại:** Runtime → Change runtime type → GPU **T4** → lưu. Chạy lần lượt từng ô ▶ từ trên xuống. Token Civitai chỉ dán vào ô mật khẩu trong Colab, **đừng gửi token cho ai và đừng lưu token vào notebook**. NTR MIX, LoRA Watercolor và LoRA Tite Kubo đều cần token mới tải được.

Cách lấy token: vào [civitai.com/user/account](https://civitai.com/user/account) → **Security & Apps** (hoặc tìm "API") → **API Keys** → **Add API key**, sao chép token (chỉ hiện một lần).

**Bản sao trên Google Drive không tự cập nhật.** Hãy mở lại [link Colab trên GitHub](https://colab.research.google.com/github/hoangnguyencccc194-blip/rpchat-cd8e0282/blob/main/colab/comfyui_bleach.ipynb), bấm **Sao chép vào Drive**, rồi chạy bản mới.


In [ ]:
#@title Giữ Colab không bị tắt khi để nền (nên chạy trên điện thoại)
%%html
<p>Nhấn phát đoạn im lặng này rồi để tab Colab mở khi đang tạo ảnh.</p>
<audio src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" controls loop></audio>


In [ ]:
#@title 1) Cài đặt (chạy ô này trước)
import ipywidgets as w
from IPython.display import display, HTML, Markdown

# Bộ nhớ trong phiên Colab thôi, không ghi token ra file hay in ra màn hình.
if "BLEACH_CFG" not in globals():
    BLEACH_CFG = {}

_token = w.Password(
    placeholder="Dán API token Civitai vào đây",
    layout=w.Layout(width="100%"),
)
_ckpt = w.Dropdown(
    options=[
        ("NTR MIX XIII (mặc định, nét minh họa)", "ntr_xiii"),
        ("WAI-illustrious v15 (bộ cũ)", "wai_v15"),
    ],
    value="ntr_xiii",
    description="Model nền:",
    style={"description_width": "110px"},
    layout=w.Layout(width="100%"),
)
_preset = w.Dropdown(
    options=[
        ("Rangiku (bleach_rangiku)", "rangiku"),
        ("Orihime (bleach_orihime)", "orihime"),
    ],
    value="rangiku",
    description="Mở sẵn:",
    style={"description_width": "110px"},
    layout=w.Layout(width="100%"),
)
_drive = w.Checkbox(
    value=False,
    description="Lưu model trên Google Drive (lần sau khỏi tải lại)",
    indent=False,
    layout=w.Layout(width="100%"),
)
_use_water = w.Checkbox(
    value=True,
    description="LoRA Bleach Watercolor  (từ kích hoạt bleach_watercolor, ~109 MB)",
    indent=False,
    layout=w.Layout(width="100%"),
)
_w_water = w.BoundedFloatText(
    value=0.85, min=0.0, max=2.0, step=0.05,
    description="Weight:",
    style={"description_width": "80px"},
    layout=w.Layout(width="220px"),
)
_use_kubo = w.Checkbox(
    value=True,
    description="LoRA Tite Kubo Style  (từ kích hoạt Tite_kubo_style, nặng 1,27 GB — tắt nếu T4 hết RAM)",
    indent=False,
    layout=w.Layout(width="100%"),
)
_w_kubo = w.BoundedFloatText(
    value=0.55, min=0.0, max=2.0, step=0.05,
    description="Weight:",
    style={"description_width": "80px"},
    layout=w.Layout(width="220px"),
)
_use_rangiku = w.Checkbox(
    value=True,
    description="LoRA nhân vật Rangiku  (từ kích hoạt rgku, ~55 MB)",
    indent=False,
    layout=w.Layout(width="100%"),
)
_w_rangiku = w.BoundedFloatText(
    value=0.85, min=0.0, max=2.0, step=0.05,
    description="Weight:",
    style={"description_width": "80px"},
    layout=w.Layout(width="220px"),
)
_extra = w.Textarea(
    placeholder="Link LoRA Civitai thêm (không bắt buộc), mỗi dòng một link trang hoặc ?modelVersionId=...",
    layout=w.Layout(width="100%", height="70px"),
)
_status = w.HTML("")
_btn = w.Button(description="Lưu cài đặt", button_style="success", layout=w.Layout(width="200px", height="40px"))

def _save(_=None):
    tok = (_token.value or "").strip()
    if not tok:
        _status.value = '<p style="color:#b00020"><b>Thiếu token Civitai.</b> Dán token vào ô mật khẩu rồi bấm Lưu cài đặt.</p>'
        return
    BLEACH_CFG["token"] = tok
    BLEACH_CFG["use_drive"] = bool(_drive.value)
    BLEACH_CFG["ckpt_key"] = _ckpt.value
    BLEACH_CFG["default_preset"] = _preset.value
    BLEACH_CFG["use_watercolor"] = bool(_use_water.value)
    BLEACH_CFG["w_watercolor"] = float(_w_water.value)
    BLEACH_CFG["use_kubo"] = bool(_use_kubo.value)
    BLEACH_CFG["w_kubo"] = float(_w_kubo.value)
    BLEACH_CFG["use_rangiku"] = bool(_use_rangiku.value)
    BLEACH_CFG["w_rangiku"] = float(_w_rangiku.value)
    extra_lines = [ln.strip() for ln in (_extra.value or "").splitlines() if ln.strip()]
    BLEACH_CFG["extra_lora_urls"] = extra_lines
    _token.value = ""  # xóa khỏi ô, tránh bị lưu vào file notebook
    ckpt_label = "NTR MIX XIII" if _ckpt.value == "ntr_xiii" else "WAI-illustrious v15"
    loras = []
    if BLEACH_CFG["use_watercolor"]:
        loras.append(f"Watercolor {BLEACH_CFG['w_watercolor']}")
    if BLEACH_CFG["use_kubo"]:
        loras.append(f"Tite Kubo Style {BLEACH_CFG['w_kubo']}")
    if BLEACH_CFG["use_rangiku"]:
        loras.append(f"Rangiku rgku {BLEACH_CFG['w_rangiku']}")
    _status.value = (
        f'<p style="color:#0a7a2f"><b>Đã lưu cài đặt.</b> Token được giữ trong phiên này, không in ra. '
        f'Model: {ckpt_label}. Mở sẵn: {_preset.value}. '
        f'LoRA sẽ tải: {", ".join(loras) if loras else "không có"}. '
        f'Google Drive: {"bật" if BLEACH_CFG["use_drive"] else "tắt"}. '
        f'LoRA thêm: {len(extra_lines)}.</p>'
    )

_btn.on_click(_save)
display(HTML("<h3>Cài đặt</h3><p>Token Civitai (ô ẩn ký tự, chỉ nhập trong Colab):</p>"))
display(_token, _ckpt, _preset, _drive)
display(HTML("<p><b>LoRA có sẵn</b> (bỏ tick thì không tải, không nạp — đỡ nặng T4):</p>"))
display(_use_water, _w_water, _use_kubo, _w_kubo, _use_rangiku, _w_rangiku)
display(HTML("<p>LoRA thêm (không bắt buộc, dán link trang Civitai cũng được):</p>"), _extra, _btn, _status)
display(Markdown(
    "Khuyên dùng: Watercolor **0.7–1.0**, Tite Kubo Style **0.55**, Rangiku **0.8–1.0**. "
    "Sau khi lưu, chạy ô **2) Cài đặt và khởi động ComfyUI**."
))


In [ ]:
#@title 2) Cài đặt ComfyUI, tải model đã chọn và mở link
import os, re, sys, json, time, shutil, stat, tempfile, subprocess, urllib.parse
from pathlib import Path
from IPython.display import display, HTML, Markdown

def _die(msg):
    display(HTML(f'<p style="color:#b00020;font-size:1.05rem"><b>{msg}</b></p>'))
    raise SystemExit(msg)

if "BLEACH_CFG" not in globals() or not BLEACH_CFG.get("token"):
    _die("Bạn chưa lưu token Civitai. Hãy chạy ô 1) Cài đặt, dán token, bấm Lưu cài đặt, rồi chạy lại ô này.")

TOKEN = BLEACH_CFG["token"]
USE_DRIVE = bool(BLEACH_CFG.get("use_drive"))
CKPT_KEY = BLEACH_CFG.get("ckpt_key") or "ntr_xiii"
DEFAULT_PRESET = BLEACH_CFG.get("default_preset") or "rangiku"
USE_WATER = bool(BLEACH_CFG.get("use_watercolor", True))
USE_KUBO = bool(BLEACH_CFG.get("use_kubo", True))
USE_RANGIKU = bool(BLEACH_CFG.get("use_rangiku", True))
W_WATER = float(BLEACH_CFG.get("w_watercolor", 0.85))
W_KUBO = float(BLEACH_CFG.get("w_kubo", 0.55))
W_RANGIKU = float(BLEACH_CFG.get("w_rangiku", 0.85))
EXTRA_URLS = list(BLEACH_CFG.get("extra_lora_urls") or [])

CHECKPOINTS = {
    "ntr_xiii": {
        "version_id": 1166878,
        "filename": "NTRMIX XIII2.safetensors",
        "min_bytes": 6_000_000_000,
        "label": "NTR MIX XIII (~6,5 GB)",
    },
    "wai_v15": {
        "version_id": 2167369,
        "filename": "WAI-Nsfw-Illustrious-15.safetensors",
        "min_bytes": 6_000_000_000,
        "label": "WAI-illustrious v15 (~6,5 GB)",
    },
}
LORA_CATALOG = {
    "watercolor": {
        "version_id": 2744203,
        "filename": "bleach_watercolor.safetensors",
        "min_bytes": 50_000_000,
        "label": "LoRA Bleach Watercolor (~109 MB)",
        "trigger": "bleach_watercolor",
        "weight_key": "w_watercolor",
    },
    "kubo": {
        "version_id": 2260426,
        "filename": "Tite_Kubo_Style_lora__drip_art.safetensors",
        "min_bytes": 1_000_000_000,
        "label": "LoRA Tite Kubo Style (~1,27 GB)",
        "trigger": "Tite_kubo_style",
        "weight_key": "w_kubo",
    },
    "rangiku": {
        "version_id": 1595729,
        "filename": "matsumotorangiku-illu-nvwls-v1.safetensors",
        "min_bytes": 40_000_000,
        "label": "LoRA Rangiku rgku (~55 MB)",
        "trigger": "rgku",
        "weight_key": "w_rangiku",
    },
}

if CKPT_KEY not in CHECKPOINTS:
    _die("Model nền không hợp lệ. Chạy lại ô 1 rồi chọn NTR MIX XIII hoặc WAI v15.")
CKPT = CHECKPOINTS[CKPT_KEY]
CKPT_NAME = CKPT["filename"]

IN_COLAB = Path("/content").exists()
ROOT = Path("/content" if IN_COLAB else "/tmp/comfy_runtime")
COMFY = ROOT / "ComfyUI"
DRIVE_ROOT = Path("/content/drive/MyDrive/rpchat_comfyui")

# ---- Google Drive (tùy chọn) ----
if USE_DRIVE:
    if not IN_COLAB:
        _die("Google Drive chỉ dùng được khi chạy trên Google Colab.")
    try:
        from google.colab import drive
        display(Markdown("Đang gắn Google Drive…"))
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        _die("Không gắn được Google Drive. Bỏ tick ô Drive ở bước 1 rồi chạy lại, hoặc cho phép Colab truy cập Drive.")
    (DRIVE_ROOT / "models" / "checkpoints").mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / "models" / "loras").mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / "output").mkdir(parents=True, exist_ok=True)

# ---- Cài gói cần thiết, không đụng bản PyTorch sẵn có trên Colab ----
display(Markdown("Đang cài công cụ tải và sao chép ComfyUI…"))
subprocess.run(["apt-get", "update", "-qq"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2", "wget"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if not (COMFY / "main.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY)], check=True)

req_path = COMFY / "requirements.txt"
skip = {"torch", "torchvision", "torchaudio"}
filtered = []
for line in req_path.read_text().splitlines():
    raw = line.strip()
    if not raw or raw.startswith("#"):
        continue
    name = re.split(r"[=<>\[]", raw, maxsplit=1)[0].strip().lower()
    if name not in skip:
        filtered.append(raw)
if filtered:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *filtered], check=True)

# ---- Thư mục model (symlink sang Drive nếu bật) ----
ckpt_dir = COMFY / "models" / "checkpoints"
lora_dir = COMFY / "models" / "loras"
ckpt_dir.mkdir(parents=True, exist_ok=True)
lora_dir.mkdir(parents=True, exist_ok=True)
if USE_DRIVE:
    def _link(src: Path, dest: Path):
        if dest.is_symlink() or dest.exists():
            if dest.is_dir() and not dest.is_symlink():
                for p in dest.glob("*"):
                    target = src / p.name
                    if not target.exists():
                        shutil.move(str(p), str(target))
                shutil.rmtree(dest)
            else:
                dest.unlink()
        dest.symlink_to(src, target_is_directory=True)
    _link(DRIVE_ROOT / "models" / "checkpoints", ckpt_dir)
    _link(DRIVE_ROOT / "models" / "loras", lora_dir)

ckpt_path = ckpt_dir / CKPT_NAME

# ---- Tải từ Civitai: token chỉ nằm trong file tạm 0600, không in, không đưa vào argv ----
def _civitai_ok(version_id: int, token: str) -> tuple:
    import urllib.request, urllib.error
    url = f"https://civitai.com/api/download/models/{version_id}"
    req = urllib.request.Request(url, method="GET", headers={
        "Authorization": f"Bearer {token}",
        "User-Agent": "rpchat-comfyui-colab",
    })
    try:
        class NoRedirect(urllib.request.HTTPRedirectHandler):
            def redirect_request(self, req, fp, code, msg, headers, newurl):
                return None
        opener = urllib.request.build_opener(NoRedirect)
        try:
            resp = opener.open(req, timeout=45)
            code = getattr(resp, "status", 200)
            loc = ""
            resp.close()
        except urllib.error.HTTPError as e:
            code = e.code
            loc = e.headers.get("Location") or ""
        if code in (200, 206):
            return True, ""
        if code in (301, 302, 303, 307, 308):
            low = loc.lower()
            if "login" in low or "reason=download-auth" in low:
                return False, "invalid"
            if loc:
                return True, ""
        if code in (401, 403):
            return False, "invalid"
        return False, f"http_{code}"
    except Exception:
        return False, "network"

def _download_civitai(version_id: int, dest: Path, token: str, min_bytes: int, label: str):
    if dest.exists() and dest.stat().st_size >= min_bytes:
        mb = dest.stat().st_size / 1024 / 1024
        display(Markdown(f"Đã có **{dest.name}** ({mb:.1f} MB), bỏ qua tải."))
        return
    ok, why = _civitai_ok(version_id, token)
    if not ok:
        if why == "invalid":
            _die("Token Civitai thiếu hoặc không hợp lệ, hoặc không đủ quyền tải file này. Kiểm tra lại token ở civitai.com/user/account rồi lưu ở ô 1.")
        if why == "network":
            _die("Không kết nối được tới Civitai. Thử chạy lại ô này sau.")
        _die(f"Civitai từ chối tải {label} (mã {why}). Thử lại sau hoặc kiểm tra token.")
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    display(Markdown(f"Đang tải **{label}** về `{dest.name}`… Có thể mất vài phút."))
    cfg = Path(tempfile.mkstemp(prefix="civitai_", suffix=".cfg")[1])
    try:
        os.chmod(cfg, 0o600)
        cfg.write_text(
            "location\n"
            "fail\n"
            "silent\n"
            "show-error\n"
            "continue-at -\n"
            f"output = \"{part}\"\n"
            f"url = \"https://civitai.com/api/download/models/{version_id}?token={token}\"\n"
        )
        os.chmod(cfg, 0o600)
        r = subprocess.run(["curl", "--config", str(cfg)], capture_output=True)
        if r.returncode != 0 or not part.exists() or part.stat().st_size < min_bytes:
            if shutil.which("aria2c"):
                inp = Path(tempfile.mkstemp(prefix="aria_", suffix=".txt")[1])
                try:
                    os.chmod(inp, 0o600)
                    inp.write_text(f"https://civitai.com/api/download/models/{version_id}?token={token}\n out={part.name}\n dir={part.parent}\n")
                    os.chmod(inp, 0o600)
                    r2 = subprocess.run(
                        ["aria2c", "-c", "-x", "16", "-s", "16", "--console-log-level=error", "-i", str(inp)],
                        capture_output=True,
                    )
                    if r2.returncode != 0:
                        _die("Tải file thất bại. Token có thể không hợp lệ, hoặc mạng bị ngắt. Thử chạy lại ô này.")
                finally:
                    try:
                        inp.unlink(missing_ok=True)
                    except Exception:
                        pass
            else:
                _die("Tải file thất bại. Token có thể không hợp lệ, hoặc mạng bị ngắt. Thử chạy lại ô này.")
        if not part.exists() or part.stat().st_size < min_bytes:
            if part.exists():
                part.unlink()
            _die(f"File {label} tải không đủ dung lượng. Xóa file dở rồi chạy lại ô này. Kiểm tra token nếu lỗi lặp lại.")
        part.replace(dest)
        display(Markdown(f"Đã tải xong **{dest.name}** ({dest.stat().st_size/1024/1024:.1f} MB)."))
    finally:
        try:
            cfg.unlink(missing_ok=True)
        except Exception:
            pass

def _parse_version_id(text: str):
    text = text.strip()
    m = re.search(r"modelVersionId=(\d+)", text)
    if m:
        return int(m.group(1))
    m = re.search(r"/api/download/models/(\d+)", text)
    if m:
        return int(m.group(1))
    if re.fullmatch(r"\d+", text):
        return int(text)
    m = re.search(r"civitai\.com/models/(\d+)", text)
    if m:
        import urllib.request
        mid = m.group(1)
        try:
            with urllib.request.urlopen(f"https://civitai.com/api/v1/models/{mid}", timeout=45) as resp:
                data = json.loads(resp.read().decode())
            vers = data.get("modelVersions") or []
            if vers:
                return int(vers[0]["id"])
        except Exception:
            return None
    return None

# Chỉ tải model đã chọn và LoRA đang bật (không tải file tắt để đỡ đầy ổ và RAM T4)
_download_civitai(CKPT["version_id"], ckpt_path, TOKEN, CKPT["min_bytes"], CKPT["label"])

enabled_style = []  # (key, filename, weight) for watercolor then kubo
if USE_WATER:
    meta = LORA_CATALOG["watercolor"]
    _download_civitai(meta["version_id"], lora_dir / meta["filename"], TOKEN, meta["min_bytes"], meta["label"])
    enabled_style.append(("watercolor", meta["filename"], W_WATER))
if USE_KUBO:
    meta = LORA_CATALOG["kubo"]
    _download_civitai(meta["version_id"], lora_dir / meta["filename"], TOKEN, meta["min_bytes"], meta["label"])
    enabled_style.append(("kubo", meta["filename"], W_KUBO))
rangiku_file = None
if USE_RANGIKU:
    meta = LORA_CATALOG["rangiku"]
    _download_civitai(meta["version_id"], lora_dir / meta["filename"], TOKEN, meta["min_bytes"], meta["label"])
    rangiku_file = meta["filename"]

extra_files = []
for i, raw in enumerate(EXTRA_URLS, start=1):
    vid = _parse_version_id(raw)
    if not vid:
        display(HTML(f'<p style="color:#b00020">Bỏ qua LoRA thêm không đọc được: (dòng {i}). Cần link Civitai có modelVersionId hoặc id phiên bản.</p>'))
        continue
    fname = f"extra_lora_{i}.safetensors"
    dest = lora_dir / fname
    _download_civitai(vid, dest, TOKEN, 1_000_000, f"LoRA thêm #{i}")
    extra_files.append(fname)

BLEACH_CFG["ckpt_name"] = CKPT_NAME
BLEACH_CFG["extra_lora_files"] = extra_files

# ---- Prompts ----
STYLE_CORE = (
    "masterpiece, best quality, amazing quality, official art, traditional media, "
    "colored pencil (medium), watercolor (medium), sketch lines, (artist:kubo tite), source manga"
)
def _style_triggers():
    parts = []
    if USE_WATER:
        parts.append("bleach_watercolor")
    if USE_KUBO:
        parts.append("Tite_kubo_style")
    return (", " + ", ".join(parts)) if parts else ""

POS_ORIHIME = (
    STYLE_CORE + _style_triggers() + ", 1girl, solo, inoue orihime, bleach, long hair, "
    "orange hair, brown eyes, parted bangs, hair ornament, hair flower, blue flower, "
    "green shirt, frilled shirt, short sleeves, blue jeans, white background, "
    "simple background, cowboy shot"
)
# Từ kích hoạt thật của LoRA 1411559 / 1595729: rgku, kèm tag diện mạo/trang phục trên thẻ
# (bỏ "huge breasts, cleavage" cho prompt mặc định nghiêng SFW, vẫn giữ black kimono + pink scarf).
POS_RANGIKU = (
    STYLE_CORE + _style_triggers() + ", 1girl, solo"
    + (", rgku" if USE_RANGIKU else "")
    + ", matsumoto rangiku, bleach, orange hair, mole under mouth, blue eyes, long hair, "
    "lips, collarbone, chain necklace, black kimono, pink scarf, white sash, wide sleeves, "
    "japanese clothes, smile, white background, simple background, cowboy shot"
)
NEG = (
    "bad quality, worst quality, worst detail, nsfw, explicit, monochrome, "
    "greyscale, 3d, realistic, english text, watermark, signature, censor"
)
WIDTH, HEIGHT, STEPS, CFG = 832, 1216, 30, 5.0

def _loras_for(preset: str):
    """Chỉ móc LoRA đã tải. Orihime không nạp LoRA nhân vật Rangiku."""
    items = list(enabled_style)  # watercolor, then Tite Kubo
    if preset == "rangiku" and rangiku_file:
        items.append(("rangiku", rangiku_file, W_RANGIKU))
    for fname in extra_files:
        items.append(("extra", fname, W_WATER if W_WATER else 0.8))
    return [(fn, wt) for _k, fn, wt in items]

def build_api_workflow(ckpt_name, loras, pos, neg, seed=0, width=WIDTH, height=HEIGHT,
                       steps=STEPS, cfg=CFG, batch=1, prefix="bleach"):
    g = {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": ckpt_name}},
        "5": {"class_type": "EmptyLatentImage", "inputs": {
            "width": int(width), "height": int(height), "batch_size": int(batch)}},
    }
    prev = "1"
    last_lora = None
    for i, (fname, weight) in enumerate(loras):
        nid = str(10 + i)
        g[nid] = {"class_type": "LoraLoader", "inputs": {
            "model": [prev, 0], "clip": [prev, 1], "lora_name": fname,
            "strength_model": float(weight), "strength_clip": float(weight)}}
        prev = nid
        last_lora = nid
    clip_ref = [last_lora, 1] if last_lora else ["1", 1]
    model_ref = [last_lora, 0] if last_lora else ["1", 0]
    g["6"] = {"class_type": "CLIPTextEncode", "inputs": {"text": pos, "clip": clip_ref}}
    g["7"] = {"class_type": "CLIPTextEncode", "inputs": {"text": neg, "clip": clip_ref}}
    g["3"] = {"class_type": "KSampler", "inputs": {
        "seed": int(seed), "steps": int(steps), "cfg": float(cfg),
        "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0,
        "model": model_ref, "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0]}}
    g["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["1", 2]}}
    g["9"] = {"class_type": "SaveImage", "inputs": {"filename_prefix": prefix, "images": ["8", 0]}}
    return g

def build_ui_workflow(ckpt_name, loras, pos, neg, prefix="bleach"):
    nodes = []
    links = []
    lid = 0

    def add_link(frm, fslot, to, tslot, typ):
        nonlocal lid
        lid += 1
        links.append([lid, frm, fslot, to, tslot, typ])
        return lid

    def node(nid):
        for n in nodes:
            if n["id"] == nid:
                return n
        raise KeyError(nid)

    def set_out(nid, slot, link_id):
        n = node(nid)
        ls = n["outputs"][slot].get("links")
        if ls is None:
            n["outputs"][slot]["links"] = [link_id]
        else:
            ls.append(link_id)

    def set_in(nid, slot, link_id):
        node(nid)["inputs"][slot]["link"] = link_id

    nodes.append({
        "id": 1, "type": "CheckpointLoaderSimple", "pos": [40, 220], "size": [380, 98],
        "flags": {}, "order": 0, "mode": 0, "inputs": [],
        "outputs": [
            {"name": "MODEL", "type": "MODEL", "links": [], "slot_index": 0},
            {"name": "CLIP", "type": "CLIP", "links": [], "slot_index": 1},
            {"name": "VAE", "type": "VAE", "links": [], "slot_index": 2},
        ],
        "properties": {"Node name for S&R": "CheckpointLoaderSimple"},
        "widgets_values": [ckpt_name],
    })
    model_src, model_slot = 1, 0
    clip_src, clip_slot = 1, 1
    nid = 2
    x = 450
    for fname, weight in loras:
        nodes.append({
            "id": nid, "type": "LoraLoader", "pos": [x, 220], "size": [340, 126],
            "flags": {}, "order": nid - 1, "mode": 0,
            "inputs": [
                {"name": "model", "type": "MODEL", "link": None},
                {"name": "clip", "type": "CLIP", "link": None},
            ],
            "outputs": [
                {"name": "MODEL", "type": "MODEL", "links": [], "slot_index": 0},
                {"name": "CLIP", "type": "CLIP", "links": [], "slot_index": 1},
            ],
            "properties": {"Node name for S&R": "LoraLoader"},
            "widgets_values": [fname, float(weight), float(weight)],
        })
        a = add_link(model_src, model_slot, nid, 0, "MODEL")
        b = add_link(clip_src, clip_slot, nid, 1, "CLIP")
        set_out(model_src, model_slot, a)
        set_out(clip_src, clip_slot, b)
        set_in(nid, 0, a)
        set_in(nid, 1, b)
        model_src, model_slot = nid, 0
        clip_src, clip_slot = nid, 1
        nid += 1
        x += 370

    pos_id, neg_id, lat_id = nid, nid + 1, nid + 2
    sam_id, vae_id, save_id = nid + 3, nid + 4, nid + 5
    px = x + 20
    nodes.append({
        "id": pos_id, "type": "CLIPTextEncode", "pos": [px, 40], "size": [400, 200],
        "flags": {}, "order": pos_id - 1, "mode": 0, "title": "Positive",
        "inputs": [{"name": "clip", "type": "CLIP", "link": None}],
        "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [], "slot_index": 0}],
        "properties": {"Node name for S&R": "CLIPTextEncode"},
        "widgets_values": [pos],
    })
    nodes.append({
        "id": neg_id, "type": "CLIPTextEncode", "pos": [px, 280], "size": [400, 160],
        "flags": {}, "order": neg_id - 1, "mode": 0, "title": "Negative",
        "inputs": [{"name": "clip", "type": "CLIP", "link": None}],
        "outputs": [{"name": "CONDITIONING", "type": "CONDITIONING", "links": [], "slot_index": 0}],
        "properties": {"Node name for S&R": "CLIPTextEncode"},
        "widgets_values": [neg],
    })
    nodes.append({
        "id": lat_id, "type": "EmptyLatentImage", "pos": [px, 500], "size": [280, 106],
        "flags": {}, "order": lat_id - 1, "mode": 0, "inputs": [],
        "outputs": [{"name": "LATENT", "type": "LATENT", "links": [], "slot_index": 0}],
        "properties": {"Node name for S&R": "EmptyLatentImage"},
        "widgets_values": [WIDTH, HEIGHT, 1],
    })
    lp = add_link(clip_src, clip_slot, pos_id, 0, "CLIP")
    ln = add_link(clip_src, clip_slot, neg_id, 0, "CLIP")
    set_out(clip_src, clip_slot, lp)
    set_out(clip_src, clip_slot, ln)
    set_in(pos_id, 0, lp)
    set_in(neg_id, 0, ln)

    sx = px + 440
    nodes.append({
        "id": sam_id, "type": "KSampler", "pos": [sx, 200], "size": [300, 262],
        "flags": {}, "order": sam_id - 1, "mode": 0,
        "inputs": [
            {"name": "model", "type": "MODEL", "link": None},
            {"name": "positive", "type": "CONDITIONING", "link": None},
            {"name": "negative", "type": "CONDITIONING", "link": None},
            {"name": "latent_image", "type": "LATENT", "link": None},
        ],
        "outputs": [{"name": "LATENT", "type": "LATENT", "links": [], "slot_index": 0}],
        "properties": {"Node name for S&R": "KSampler"},
        "widgets_values": [0, "randomize", STEPS, CFG, "euler_ancestral", "normal", 1.0],
    })
    lm = add_link(model_src, model_slot, sam_id, 0, "MODEL")
    lpos = add_link(pos_id, 0, sam_id, 1, "CONDITIONING")
    lneg = add_link(neg_id, 0, sam_id, 2, "CONDITIONING")
    llat = add_link(lat_id, 0, sam_id, 3, "LATENT")
    set_out(model_src, model_slot, lm)
    set_out(pos_id, 0, lpos)
    set_out(neg_id, 0, lneg)
    set_out(lat_id, 0, llat)
    set_in(sam_id, 0, lm)
    set_in(sam_id, 1, lpos)
    set_in(sam_id, 2, lneg)
    set_in(sam_id, 3, llat)

    vx = sx + 340
    nodes.append({
        "id": vae_id, "type": "VAEDecode", "pos": [vx, 200], "size": [240, 46],
        "flags": {}, "order": vae_id - 1, "mode": 0,
        "inputs": [
            {"name": "samples", "type": "LATENT", "link": None},
            {"name": "vae", "type": "VAE", "link": None},
        ],
        "outputs": [{"name": "IMAGE", "type": "IMAGE", "links": [], "slot_index": 0}],
        "properties": {"Node name for S&R": "VAEDecode"},
        "widgets_values": [],
    })
    ls = add_link(sam_id, 0, vae_id, 0, "LATENT")
    lv = add_link(1, 2, vae_id, 1, "VAE")
    set_out(sam_id, 0, ls)
    set_out(1, 2, lv)
    set_in(vae_id, 0, ls)
    set_in(vae_id, 1, lv)

    nodes.append({
        "id": save_id, "type": "SaveImage", "pos": [vx + 280, 200], "size": [300, 260],
        "flags": {}, "order": save_id - 1, "mode": 0,
        "inputs": [{"name": "images", "type": "IMAGE", "link": None}],
        "outputs": [{"name": "images", "type": "IMAGE", "links": None, "slot_index": 0}],
        "properties": {"Node name for S&R": "SaveImage"},
        "widgets_values": [prefix],
    })
    li = add_link(vae_id, 0, save_id, 0, "IMAGE")
    set_out(vae_id, 0, li)
    set_in(save_id, 0, li)

    return {
        "last_node_id": save_id,
        "last_link_id": lid,
        "nodes": nodes,
        "links": links,
        "groups": [],
        "config": {},
        "extra": {"ds": {"scale": 0.8, "offset": [0, 0]}, "workflow_name": prefix},
        "version": 0.4,
    }

loras_orihime = _loras_for("orihime")
loras_rangiku = _loras_for("rangiku")
ui_orihime = build_ui_workflow(CKPT_NAME, loras_orihime, POS_ORIHIME, NEG, "bleach_orihime")
ui_rangiku = build_ui_workflow(CKPT_NAME, loras_rangiku, POS_RANGIKU, NEG, "bleach_rangiku")
api_orihime = build_api_workflow(CKPT_NAME, loras_orihime, POS_ORIHIME, NEG, prefix="bleach_orihime")
api_rangiku = build_api_workflow(CKPT_NAME, loras_rangiku, POS_RANGIKU, NEG, prefix="bleach_rangiku")

ui_default = ui_rangiku if DEFAULT_PRESET == "rangiku" else ui_orihime
(COMFY / "default_workflow.json").write_text(json.dumps(ui_default, ensure_ascii=False, indent=2))
wf_dir = COMFY / "user" / "default" / "workflows"
wf_dir.mkdir(parents=True, exist_ok=True)
(wf_dir / "bleach_orihime.json").write_text(json.dumps(ui_orihime, ensure_ascii=False, indent=2))
(wf_dir / "bleach_rangiku.json").write_text(json.dumps(ui_rangiku, ensure_ascii=False, indent=2))
(ROOT / "bleach_orihime_workflow.json").write_text(json.dumps(ui_orihime, ensure_ascii=False, indent=2))
(ROOT / "bleach_rangiku_workflow.json").write_text(json.dumps(ui_rangiku, ensure_ascii=False, indent=2))
(ROOT / "bleach_orihime_api.json").write_text(json.dumps(api_orihime, ensure_ascii=False, indent=2))
(ROOT / "bleach_rangiku_api.json").write_text(json.dumps(api_rangiku, ensure_ascii=False, indent=2))

BLEACH_CFG["comfy"] = str(COMFY)
BLEACH_CFG["pos_orihime"] = POS_ORIHIME
BLEACH_CFG["pos_rangiku"] = POS_RANGIKU
BLEACH_CFG["neg"] = NEG
BLEACH_CFG["loras_orihime"] = loras_orihime
BLEACH_CFG["loras_rangiku"] = loras_rangiku
BLEACH_CFG["default_preset"] = DEFAULT_PRESET
display(Markdown(
    "Đã ghi hai workflow: `bleach_rangiku` (LoRA nét + LoRA nhân vật, nếu bạn đã bật) "
    "và `bleach_orihime` (không nạp LoRA Rangiku). "
    f"Giao diện sẽ mở sẵn **bleach_{DEFAULT_PRESET}**."
))

# ---- cloudflared ----
cf_bin = ROOT / "cloudflared"
if not cf_bin.exists():
    subprocess.run([
        "wget", "-q", "-O", str(cf_bin),
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    ], check=True)
    cf_bin.chmod(cf_bin.stat().st_mode | stat.S_IEXEC)

# ---- Khởi động ComfyUI ----
for proc_name in ("main.py", "cloudflared"):
    subprocess.run(["pkill", "-f", f"{COMFY}.*main.py" if proc_name == "main.py" else str(cf_bin)], check=False)

log_path = ROOT / "comfyui.log"
cf_log = ROOT / "cloudflared.log"
log_f = open(log_path, "w")
comfy_proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
     "--preview-method", "auto", "--enable-cors-header"],
    cwd=str(COMFY),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env={**os.environ, "HF_HUB_DISABLE_TELEMETRY": "1"},
)
BLEACH_CFG["comfy_proc"] = comfy_proc

import urllib.request
ready = False
for _ in range(90):
    if comfy_proc.poll() is not None:
        tail = log_path.read_text(errors="ignore")[-2000:]
        _die("ComfyUI thoát sớm. Một đoạn log (không chứa token): " + tail.replace("<", " "))
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(2)
if not ready:
    _die("ComfyUI không sẵn sàng sau khoảng 3 phút. Chạy lại ô này, hoặc xem runtime đã chọn GPU T4 chưa.")

cf_f = open(cf_log, "w")
cf_proc = subprocess.Popen(
    [str(cf_bin), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"],
    stdout=cf_f,
    stderr=subprocess.STDOUT,
)
BLEACH_CFG["cf_proc"] = cf_proc

url = None
for _ in range(40):
    txt = cf_log.read_text(errors="ignore") if cf_log.exists() else ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        url = m.group(0)
        break
    time.sleep(1)
if not url:
    _die("Chưa lấy được link trycloudflare. Chạy lại ô này. Nếu Colab hỏi cấp quyền mạng thì hãy cho phép.")

BLEACH_CFG["public_url"] = url
display(HTML(f"""
<div style="border:2px solid #0a7a2f;padding:16px;border-radius:12px;margin:12px 0">
  <p style="margin:0 0 8px;font-size:1.1rem"><b>ComfyUI đã sẵn sàng.</b> Mở link này trên điện thoại:</p>
  <p style="margin:0;font-size:1.2rem"><a href="{url}" target="_blank">{url}</a></p>
  <p style="margin:8px 0 0;color:#333">Workflow <code>bleach_{DEFAULT_PRESET}</code> được mở sẵn (model {CKPT_NAME}).
  Bấm <b>Queue</b> để ra ảnh. Đổi nhân vật: vào mục Workflows chọn <code>bleach_rangiku</code> hoặc <code>bleach_orihime</code>.</p>
</div>
"""))
display(Markdown("Muốn tạo ảnh ngay trong Colab, kéo xuống ô **Chế độ đơn giản** rồi bấm tạo." ))


## 3) Chế độ đơn giản (dùng trên điện thoại, không cần vào giao diện ComfyUI)

Chạy ô dưới sau khi ô 2 đã hiện link. Chọn sẵn **Rangiku** hoặc **Orihime** để điền prompt, rồi bấm **Tạo ảnh**. Ảnh hiện ngay trong notebook; nếu bạn bật Google Drive thì ảnh cũng được lưu vào `MyDrive/rpchat_comfyui/output`. Ô này dùng đúng model và các LoRA bạn đã bật ở ô 1 (không nạp LoRA đã tắt).


In [ ]:
#@title 3) Chế độ đơn giản — chọn nhân vật, gõ prompt và tạo ảnh
import json, time, random, shutil, urllib.request, urllib.parse
from pathlib import Path
from IPython.display import display, HTML, Image as IPyImage
import ipywidgets as w

def _die(msg):
    display(HTML(f'<p style="color:#b00020"><b>{msg}</b></p>'))
    raise SystemExit(msg)

if "BLEACH_CFG" not in globals() or not BLEACH_CFG.get("comfy"):
    _die("Chưa khởi động ComfyUI. Chạy ô 1 rồi ô 2 trước.")

POS_ORIHIME = BLEACH_CFG.get("pos_orihime") or ""
POS_RANGIKU = BLEACH_CFG.get("pos_rangiku") or ""
NEG0 = BLEACH_CFG.get("neg") or (
    "bad quality, worst quality, worst detail, nsfw, explicit, monochrome, "
    "greyscale, 3d, realistic, english text, watermark, signature, censor"
)
CKPT_NAME = BLEACH_CFG.get("ckpt_name") or "NTRMIX XIII2.safetensors"
LORAS_ORIHIME = list(BLEACH_CFG.get("loras_orihime") or [])
LORAS_RANGIKU = list(BLEACH_CFG.get("loras_rangiku") or [])
DEFAULT_PRESET = BLEACH_CFG.get("default_preset") or "rangiku"
USE_DRIVE = bool(BLEACH_CFG.get("use_drive"))
COMFY = Path(BLEACH_CFG["comfy"])
API = "http://127.0.0.1:8188"

preset_w = w.Dropdown(
    options=[("Rangiku (rgku)", "rangiku"), ("Orihime", "orihime")],
    value=DEFAULT_PRESET if DEFAULT_PRESET in ("rangiku", "orihime") else "rangiku",
    description="Nhân vật:",
    style={"description_width": "90px"},
    layout=w.Layout(width="100%"),
)
pos_w = w.Textarea(value=POS_RANGIKU if DEFAULT_PRESET == "rangiku" else POS_ORIHIME,
                   description="", layout=w.Layout(width="100%", height="110px"))
neg_w = w.Textarea(value=NEG0, description="", layout=w.Layout(width="100%", height="70px"))
width_w = w.BoundedIntText(value=832, min=512, max=1280, step=8, description="Rộng:", style={"description_width": "70px"})
height_w = w.BoundedIntText(value=1216, min=512, max=1536, step=8, description="Cao:", style={"description_width": "70px"})
steps_w = w.BoundedIntText(value=30, min=8, max=50, description="Steps:", style={"description_width": "70px"})
cfg_w = w.BoundedFloatText(value=5.0, min=1.0, max=12.0, step=0.5, description="CFG:", style={"description_width": "70px"})
seed_w = w.IntText(value=-1, description="Seed:", style={"description_width": "70px"})
batch_w = w.BoundedIntText(value=1, min=1, max=2, description="Số ảnh:", style={"description_width": "70px"})
btn = w.Button(description="Tạo ảnh", button_style="success", layout=w.Layout(width="200px", height="42px"))
status = w.HTML("")
gallery = w.Output()

def _on_preset(change):
    if change.get("name") != "value":
        return
    if change["new"] == "rangiku":
        pos_w.value = POS_RANGIKU
    else:
        pos_w.value = POS_ORIHIME

preset_w.observe(_on_preset, names="value")

def _api_graph(seed, width, height, steps, cfg, batch, pos, neg, preset):
    loras = LORAS_RANGIKU if preset == "rangiku" else LORAS_ORIHIME
    prefix = "bleach_rangiku" if preset == "rangiku" else "bleach_orihime"
    g = {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": CKPT_NAME}},
        "5": {"class_type": "EmptyLatentImage", "inputs": {
            "width": int(width), "height": int(height), "batch_size": int(batch)}},
    }
    prev = "1"
    last_lora = None
    for i, (fname, weight) in enumerate(loras):
        nid = str(10 + i)
        g[nid] = {"class_type": "LoraLoader", "inputs": {
            "model": [prev, 0], "clip": [prev, 1], "lora_name": fname,
            "strength_model": float(weight), "strength_clip": float(weight)}}
        prev = nid
        last_lora = nid
    clip_ref = [last_lora, 1] if last_lora else ["1", 1]
    model_ref = [last_lora, 0] if last_lora else ["1", 0]
    g["6"] = {"class_type": "CLIPTextEncode", "inputs": {"text": pos, "clip": clip_ref}}
    g["7"] = {"class_type": "CLIPTextEncode", "inputs": {"text": neg, "clip": clip_ref}}
    g["3"] = {"class_type": "KSampler", "inputs": {
        "seed": int(seed), "steps": int(steps), "cfg": float(cfg),
        "sampler_name": "euler_ancestral", "scheduler": "normal", "denoise": 1.0,
        "model": model_ref, "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["5", 0]}}
    g["8"] = {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["1", 2]}}
    g["9"] = {"class_type": "SaveImage", "inputs": {"filename_prefix": prefix, "images": ["8", 0]}}
    return g

def _post(path, payload=None):
    data = None if payload is None else json.dumps(payload).encode()
    req = urllib.request.Request(API + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode())

def _get(path):
    with urllib.request.urlopen(API + path, timeout=60) as resp:
        return resp.read()

def _run(_=None):
    status.value = "<p>Đang tạo ảnh, đợi chút (lần đầu có thể lâu vì phải nạp model)…</p>"
    gallery.clear_output()
    width, height = int(width_w.value), int(height_w.value)
    if width % 8 or height % 8:
        status.value = '<p style="color:#b00020">Rộng và cao phải chia hết cho 8.</p>'
        return
    seed = int(seed_w.value)
    if seed < 0:
        seed = random.randint(0, 2**32 - 1)
    preset = preset_w.value
    graph = _api_graph(seed, width, height, int(steps_w.value), float(cfg_w.value),
                       int(batch_w.value), pos_w.value, neg_w.value, preset)
    try:
        queued = _post("/prompt", {"prompt": graph})
    except Exception:
        status.value = '<p style="color:#b00020">Không gửi được tới ComfyUI. Chạy lại ô 2 cho server kịp mở.</p>'
        return
    pid = queued.get("prompt_id")
    if not pid:
        status.value = '<p style="color:#b00020">ComfyUI không nhận prompt.</p>'
        return
    images_meta = []
    for _i in range(300):
        time.sleep(2)
        try:
            hist = json.loads(_get(f"/history/{pid}"))
        except Exception:
            continue
        item = hist.get(pid)
        if not item:
            continue
        if item.get("status", {}).get("completed") or item.get("outputs"):
            for node in (item.get("outputs") or {}).values():
                for im in node.get("images") or []:
                    images_meta.append(im)
            break
    if not images_meta:
        status.value = '<p style="color:#b00020">Hết thời gian chờ hoặc không có ảnh. Thử giảm kích thước xuống 768×1152, hoặc tắt LoRA Tite Kubo Style rồi chạy lại ô 2.</p>'
        return
    saved = []
    with gallery:
        for im in images_meta:
            q = urllib.parse.urlencode({
                "filename": im.get("filename", ""),
                "subfolder": im.get("subfolder", ""),
                "type": im.get("type", "output"),
            })
            raw = _get("/view?" + q)
            out_dir = Path("/tmp/bleach_out")
            out_dir.mkdir(parents=True, exist_ok=True)
            local = out_dir / im.get("filename", "out.png")
            local.write_bytes(raw)
            if USE_DRIVE:
                dest_dir = Path("/content/drive/MyDrive/rpchat_comfyui/output")
                dest_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(local, dest_dir / local.name)
                saved.append(str(dest_dir / local.name))
            display(IPyImage(data=raw))
    msg = f"Xong. Seed = {seed}. {len(images_meta)} ảnh. Model: {CKPT_NAME}."
    if saved:
        msg += " Đã lưu vào Google Drive/rpchat_comfyui/output."
    status.value = f'<p style="color:#0a7a2f"><b>{msg}</b></p>'

btn.on_click(_run)
display(HTML(
    f"<p>Model đang dùng: <code>{CKPT_NAME}</code>. "
    "Chọn nhân vật để điền prompt và đúng chuỗi LoRA (Rangiku mới nạp LoRA <code>rgku</code>).</p>"
))
display(preset_w)
display(HTML("<p><b>Prompt dương</b></p>"), pos_w, HTML("<p><b>Prompt âm</b></p>"), neg_w)
display(w.HBox([width_w, height_w]), w.HBox([steps_w, cfg_w]), w.HBox([seed_w, batch_w]))
display(HTML("<p>Seed = <code>-1</code> thì mỗi lần một seed ngẫu nhiên.</p>"), btn, status, gallery)


**Gợi ý:** Lần tạo đầu thường lâu vì model ~6,5 GB phải nạp vào GPU. 832×1216, 30 bước, CFG 5, Euler a là mặc định mới (nét dịu hơn CFG 6). Nếu hết bộ nhớ thì tắt LoRA Tite Kubo Style (1,27 GB) rồi chạy lại ô 2, hoặc hạ ảnh còn 768×1152. Prompt Rangiku dùng từ kích hoạt `rgku` kèm `matsumoto rangiku`; Orihime dùng `inoue orihime` và `bleach_watercolor`. Khi hết phiên Colab, model trên máy ảo bị xóa — bật lưu Google Drive ở ô 1 thì lần sau khỏi tải lại. Bản sao Drive của notebook cũ không tự cập nhật, cần mở lại link GitHub rồi sao chép mới.
